# Build an eval harness with golden sets and LLM-as-judge

A team has been shipping prompt changes for six months with no regression eval. The latest change regressed accuracy on a critical use case by 14 points; nobody caught it for 3 weeks. You have 90 minutes to build the harness from scratch: a 100-example golden set with reference outputs, an LLM-as-judge with a calibrated rubric, a regression detector comparing model versions, and a CI gate that fails a pull request when the score drops more than 3 points.

The lab uses a synthetic golden set generated in the notebook (deterministic, audit-friendly). The "CI integration" runs locally in Python so the lab does not depend on a student-supplied GitHub repo.

**Duration:** about 90 minutes of work in a 120-minute session window.

**Cost preamble.** About $1.50. Two model versions x 100 examples x 2 calls (model-under-test + judge) at Sonnet rates: ~$1.20. A debug run with one extra eval pass lands around $2.20. GitHub Actions minutes are free; the lab simulates the CI gate locally.

In [ ]:
# NBVAL_SKIP
# Pinned dependencies. supabase 2.9 ships the postgrest builder we need
# for table inserts and the rpc surface.

!pip install --quiet labrun-checks==0.7.0 anthropic==0.42.0 supabase==2.9.0 pyyaml==6.0.2

In [ ]:
# Imports and per-lab constants.

import atexit
import getpass
import json
import os
import statistics
import subprocess
import sys
import textwrap
import time
import uuid
from datetime import datetime, timezone

import yaml

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupAdapter,
    CleanupResource,
)

LAB_ID = "eval-harness-golden-set-judge"
LAB_TAG_VALUE = LAB_ID

RESULTS_TABLE = "labrun_eval_harness_golden_set_judge_results"
RUNS_TABLE = "labrun_eval_harness_golden_set_judge_runs"

BASELINE_PATH = "/content/eval_baseline.json"
WORKFLOW_PATH = "/content/.github/workflows/eval.yml"

ANTHROPIC_SONNET = "claude-sonnet-4-5-20250929"
ANTHROPIC_HAIKU = "claude-haiku-4-5-20250930"

REGRESSION_THRESHOLD_POINTS = 3.0

In [ ]:
# NBVAL_SKIP
# BYOK setup: session token, Anthropic key, Supabase URL + service role key.
# GITHUB_TOKEN optional (the local CI gate does not require it).

import anthropic
from supabase import create_client, Client

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
ANTHROPIC_API_KEY = getpass.getpass("ANTHROPIC_API_KEY: ").strip()
SUPABASE_URL = getpass.getpass("SUPABASE_URL (https://xxx.supabase.co): ").strip()
SUPABASE_SERVICE_ROLE_KEY = getpass.getpass("SUPABASE_SERVICE_ROLE_KEY: ").strip()
GITHUB_TOKEN = getpass.getpass("GITHUB_TOKEN (optional; press enter to skip): ").strip()

if not all([ANTHROPIC_API_KEY, SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY]):
    print("Anthropic key + Supabase URL + service role key are all required. Re-run this cell.")
    raise SystemExit(1)

os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

anthropic_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
try:
    _ping = anthropic_client.messages.create(
        model=ANTHROPIC_HAIKU,
        max_tokens=8,
        messages=[{"role": "user", "content": "reply with: ok"}],
    )
    print(f"Anthropic credentials validated. Haiku replied: {_ping.content[0].text!r}")
except Exception as exc:
    print(f"Anthropic key rejected: {exc!r}")
    raise SystemExit(1)

supabase: Client = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)
# Cheap connectivity check via auth.
try:
    # supabase-py service-role client connects lazily; a trivial select against
    # a known table would 404 if the table is absent, which we accept here.
    # We probe the rest endpoint by listing rpc; if URL/key are bad it raises.
    _probe = supabase.table("_supabase_unlikely_table_name").select("*").limit(1)
    try:
        _probe.execute()
    except Exception:
        # 404 / table-missing is expected and proves the URL + key reach the REST API.
        pass
    print(f"Supabase reachable at {SUPABASE_URL}.")
except Exception as exc:
    print(f"Supabase rejected: {exc!r}")
    raise SystemExit(1)

register(
    session_token=session_token,
    lab_id=LAB_ID,
    credentials={"supabase_url": SUPABASE_URL},
)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, adapter, atexit safety net, orphan scan.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="local_file",
        id=WORKFLOW_PATH,
        region="local",
        cli_delete_command=f"rm -f {WORKFLOW_PATH}",
    ),
    CleanupResource(
        type="local_file",
        id=BASELINE_PATH,
        region="local",
        cli_delete_command=f"rm -f {BASELINE_PATH}",
    ),
    CleanupResource(
        type="supabase_table",
        id=RUNS_TABLE,
        region="supabase",
        cli_delete_command=f"# In Supabase SQL editor: DROP TABLE IF EXISTS public.{RUNS_TABLE}",
    ),
    CleanupResource(
        type="supabase_table",
        id=RESULTS_TABLE,
        region="supabase",
        cli_delete_command=f"# In Supabase SQL editor: DROP TABLE IF EXISTS public.{RESULTS_TABLE}",
    ),
]


class EvalHarnessCleanupAdapter:
    """Supabase tables: clear rows via the REST client; the table itself stays
    (drop requires direct Postgres access not exposed by the service-role key).
    The manifest records this; the operator drops the empty shell from the
    dashboard if they care to.

    TODO: labrun-checks 0.8 will ship a supabase adapter that supports DROP via
    Supabase Management API; until then, this inline adapter clears rows.
    """

    def delete_resource(self, credentials, resource):
        if resource.type == "local_file":
            try:
                os.remove(resource.id)
            except FileNotFoundError:
                pass
            return
        if resource.type == "supabase_table":
            try:
                # Best-effort delete-all. .neq("id", 0) is the Supabase idiom
                # because .delete() without a where-clause is rejected.
                supabase.table(resource.id).delete().neq("id", 0).execute()
            except Exception:
                pass
            return

    def describe_resource(self, credentials, resource):
        if resource.type == "local_file":
            return os.path.exists(resource.id)
        if resource.type == "supabase_table":
            try:
                rows = supabase.table(resource.id).select("id").limit(1).execute()
                return bool(rows.data)
            except Exception:
                return False
        return False

    def tag_scan(self, credentials, lab_slug, region):
        return []


CLEANUP_ADAPTER = EvalHarnessCleanupAdapter()


# Orphan scan: check local files and check if tables have rows from a prior session.
_orphans = []
for p in (WORKFLOW_PATH, BASELINE_PATH):
    if os.path.exists(p):
        _orphans.append(p)
for t in (RESULTS_TABLE, RUNS_TABLE):
    try:
        rows = supabase.table(t).select("id").limit(1).execute()
        if rows.data:
            _orphans.append(f"supabase://{t} has rows from a prior session")
    except Exception:
        pass

if _orphans:
    print("Orphan scan found leftover state from a prior session:")
    for o in _orphans:
        print(f"  {o}")
    print("Clear them (delete rows + local files) before re-running this lab.")
    raise SystemExit(1)


def _on_exit_cleanup():
    try:
        for r in list(CLEANUP_MANIFEST):
            try:
                CLEANUP_ADAPTER.delete_resource({}, r)
            except Exception:
                pass
    except Exception:
        pass


atexit.register(_on_exit_cleanup)
print("Manifest registered. Orphan scan clean.")

## Supabase table setup (read; run as-is)

The two tables this lab uses must exist before the eval harness writes to them. The lab cannot run DDL via the service-role key; instead we use the Supabase REST client to attempt inserts and let the operator create the tables via the dashboard if the inserts fail with "table not found".

The schemas are:

```sql
create table public.labrun_eval_harness_golden_set_judge_results (
  id bigserial primary key,
  run_id uuid not null,
  example_id text not null,
  score real not null,
  inserted_at timestamptz default now()
);

create table public.labrun_eval_harness_golden_set_judge_runs (
  id bigserial primary key,
  run_id uuid not null unique,
  label text not null,
  mean_score real not null,
  inserted_at timestamptz default now()
);
```

If you do not have the tables yet, open the Supabase SQL editor and run the two CREATE statements. Then continue.

In [ ]:
# NBVAL_SKIP
# Confirm the two tables exist. If not, print the SQL the operator must run.

EXPECTED_TABLES_SQL = """
-- Run these in the Supabase SQL editor before continuing the lab.

create table if not exists public.labrun_eval_harness_golden_set_judge_results (
  id bigserial primary key,
  run_id uuid not null,
  example_id text not null,
  score real not null,
  inserted_at timestamptz default now()
);

create table if not exists public.labrun_eval_harness_golden_set_judge_runs (
  id bigserial primary key,
  run_id uuid not null unique,
  label text not null,
  mean_score real not null,
  inserted_at timestamptz default now()
);
"""


def confirm_tables():
    missing = []
    for t in (RESULTS_TABLE, RUNS_TABLE):
        try:
            supabase.table(t).select("id").limit(1).execute()
        except Exception:
            missing.append(t)
    if missing:
        print("The following tables are missing. Create them in Supabase SQL editor, then re-run this cell.")
        print(EXPECTED_TABLES_SQL)
        raise SystemExit(1)
    print(f"Tables exist: {RESULTS_TABLE}, {RUNS_TABLE}.")


confirm_tables()

## Task 1: Seed the 100-example golden set

The golden set is synthetic (a fictional product called "Widget"). 100 rows of question + reference_output pairs. Insert all 100 into the `results` table with a fresh `run_id` UUID and `score=null` (we score them in Task 2).

The validator confirms 100 rows exist with the expected schema.

In [ ]:
# Task 1: synthesize the golden set, insert into the results table.

GOLDEN_SET = []
for i in range(100):
    if i % 4 == 0:
        q = f"What does Widget v{(i % 7) + 1}.0 do that v{(i % 7)}.0 did not?"
        ref = (
            f"Widget v{(i % 7) + 1}.0 adds streaming support, halves the cold-start latency, "
            f"and exposes a new /metrics endpoint."
        )
    elif i % 4 == 1:
        q = f"What is the recommended chunk size for Widget's vector store with documents under {1 + (i % 5) * 2}KB?"
        ref = f"Use 512 tokens with 50-token overlap for documents under {1 + (i % 5) * 2}KB; larger documents should bump to 1024."
    elif i % 4 == 2:
        q = f"How many concurrent requests can a Widget Standard plan handle on the {['us-east-1','us-west-2','eu-west-1','ap-south-1'][i % 4]} region?"
        ref = "Widget Standard handles 50 concurrent requests per replica; autoscaling adds replicas at 70% utilization."
    else:
        q = f"What is Widget's response to a malformed payload missing the required field 'order_id'?"
        ref = "Widget returns HTTP 422 with a JSON body listing the missing field and a link to the schema docs."
    GOLDEN_SET.append({"id": f"ex_{i:03d}", "input": q, "reference_output": ref})


def seed_golden_set():
    run_id = str(uuid.uuid4())
    # YOUR CODE: build a list of rows shaped like
    # {"run_id": run_id, "example_id": ex["id"], "score": 0.0}
    # and insert all 100 in a single supabase.table(RESULTS_TABLE).insert(rows).execute().
    raise NotImplementedError("Replace with the supabase insert.")


SEED_RUN_ID = seed_golden_set()
print(f"Seeded {len(GOLDEN_SET)} rows under run_id={SEED_RUN_ID}.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: results table has 100 rows for the seed run.


def checkpoint_1(session):
    if not isinstance(GOLDEN_SET, list) or len(GOLDEN_SET) != 100:
        return CheckpointResult(status="fail", error_reason=f"GOLDEN_SET length is {len(GOLDEN_SET)}; expected 100.")
    for r in GOLDEN_SET:
        if not all(k in r for k in ("id", "input", "reference_output")):
            return CheckpointResult(status="fail", error_reason=f"Golden row missing keys: {r}")
    try:
        rows = supabase.table(RESULTS_TABLE).select("example_id").eq("run_id", SEED_RUN_ID).execute().data
    except Exception as exc:
        return CheckpointResult(status="error", error_reason=f"Supabase select failed: {exc!r}")
    if len(rows) != 100:
        return CheckpointResult(
            status="fail",
            error_reason=f"Expected 100 result rows for run_id={SEED_RUN_ID}; found {len(rows)}.",
        )
    return CheckpointResult(status="pass")


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

`supabase.table(name).insert(list_of_dicts).execute()` is the single bulk-insert call. The service-role key bypasses RLS so you do not need a separate session.

</details>

<details><summary>Hint 2 (direction)</summary>

```
def seed_golden_set():
    run_id = str(uuid.uuid4())
    rows = [{"run_id": run_id, "example_id": ex["id"], "score": 0.0} for ex in GOLDEN_SET]
    supabase.table(RESULTS_TABLE).insert(rows).execute()
    return run_id
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2.

</details>

**Wallet check.** 100 inserts: free. The expensive part starts in Task 2 when we score with an LLM.

## Task 2: Build the LLM-as-judge with a calibrated rubric

The judge takes (input, reference_output, model_output) and returns an integer score 1-10. Uncalibrated judges cluster at 7-8 for almost everything; calibration means including 1-2 concrete example scores per level in the rubric, so the judge has anchors.

Test the judge on 10 known-good pairs (model_output == reference) and 10 known-bad pairs (model_output == "I do not know"). Average score on good must be at least 1.5 points higher than average on bad.

In [ ]:
# Task 2: calibrated judge.

JUDGE_RUBRIC = textwrap.dedent("""
You are an evaluator scoring AI assistant outputs on a 1 to 10 scale.

Score key:
- 10: matches the reference output exactly or paraphrases all facts correctly.
- 8: matches the reference on the essential facts; minor missing detail or phrasing differs.
- 6: partially correct; one important fact wrong or missing.
- 4: mostly wrong; mentions the topic but the facts are off.
- 2: refuses to answer, says "I do not know", or returns something unrelated.
- 1: hallucinates new facts not in the reference; or answers a different question.

Example scores:
- model_output 'Widget Standard handles 50 concurrent requests; autoscale at 70%.' vs reference 'Widget Standard handles 50 concurrent requests per replica; autoscaling adds replicas at 70% utilization.' -> 9.
- model_output 'I do not know.' vs reference '...' -> 2.
- model_output 'Widget Standard handles 1000 concurrent requests.' vs reference '...' -> 1 (hallucinated fact).

Reply with a single integer between 1 and 10. No other text.
""").strip()


def judge_score(prompt_input, reference, model_output):
    # YOUR CODE: call anthropic_client.messages.create with model
    # ANTHROPIC_SONNET, max_tokens=8, system=JUDGE_RUBRIC, and a user message
    # that includes input, reference, model_output. Parse the response text
    # to an int between 1 and 10; clamp out-of-range values.
    raise NotImplementedError("Replace with the judge call + parser.")


# Calibration check: 10 known-good, 10 known-bad pairs.
calibration_pairs = []
for ex in GOLDEN_SET[:10]:
    calibration_pairs.append({"input": ex["input"], "reference": ex["reference_output"], "model_output": ex["reference_output"], "label": "good"})
for ex in GOLDEN_SET[10:20]:
    calibration_pairs.append({"input": ex["input"], "reference": ex["reference_output"], "model_output": "I do not know.", "label": "bad"})

good_scores = []
bad_scores = []
for p in calibration_pairs:
    s = judge_score(p["input"], p["reference"], p["model_output"])
    if p["label"] == "good":
        good_scores.append(s)
    else:
        bad_scores.append(s)

print(f"Calibration: good mean={statistics.mean(good_scores):.2f}, bad mean={statistics.mean(bad_scores):.2f}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: good mean - bad mean >= 1.5 points.


def checkpoint_2(session):
    if not good_scores or not bad_scores:
        return CheckpointResult(status="fail", error_reason="judge_score did not produce calibration data.")
    diff = statistics.mean(good_scores) - statistics.mean(bad_scores)
    if diff < 1.5:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"good mean {statistics.mean(good_scores):.2f} - bad mean {statistics.mean(bad_scores):.2f} = {diff:.2f}; "
                f"expected >= 1.5. The rubric likely lacks concrete example scores per level; uncalibrated judges "
                f"cluster scores in a narrow band."
            ),
        )
    return CheckpointResult(status="pass")


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The judge call is the standard Sonnet messages API with `system=JUDGE_RUBRIC` and `max_tokens=8`. The parser strips whitespace and pulls the first integer it finds. Out-of-range values clamp to 1-10 so a misbehaving judge does not throw.

</details>

<details><summary>Hint 2 (direction)</summary>

```
def judge_score(prompt_input, reference, model_output):
    resp = anthropic_client.messages.create(
        model=ANTHROPIC_SONNET,
        max_tokens=8,
        system=JUDGE_RUBRIC,
        messages=[{"role": "user", "content": (
            f"INPUT: {prompt_input}\nREFERENCE: {reference}\nMODEL_OUTPUT: {model_output}\n\nScore (1-10):"
        )}],
    )
    text = resp.content[0].text.strip()
    import re as _re
    m = _re.search(r"\d+", text)
    if not m:
        return 5
    return max(1, min(10, int(m.group(0))))
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2.

</details>

**Wallet check.** 20 judge calls at Sonnet rates: roughly 5 cents. Cumulative: ~5 cents.

## Task 3: Regression detector flags the sabotaged prompt

Run the eval against the full 100 examples twice:

1. **Baseline:** the model-under-test is Sonnet with a proper system prompt ("You are a Widget expert; answer concisely with the facts.")
2. **Sabotage:** the same Sonnet with a sabotaged system prompt that produces lower-quality outputs ("Answer with at most 5 words and do not use any specific numbers.")

For each run, insert one row per example into `results` and one summary row into `runs` (with `mean_score`). Then implement `detect_regression(prev_score, curr_score, threshold=3.0) -> "block"|"allow"`. The validator queries the runs table and asserts the sabotage run scored at least 3 points below the baseline AND the detector returns "block".

To keep the cost down, the lab evaluates against 30 of the 100 examples (the full 100 lands around $3-4; the lab keeps it under $2).

In [ ]:
# Task 3: baseline + sabotage eval + detector.

EVAL_SLICE = GOLDEN_SET[:30]  # 30 of 100 to keep cost in the brief's range


def model_under_test(question, system_prompt, model=ANTHROPIC_SONNET):
    resp = anthropic_client.messages.create(
        model=model,
        max_tokens=128,
        system=system_prompt,
        messages=[{"role": "user", "content": question}],
    )
    return resp.content[0].text.strip()


BASELINE_PROMPT = "You are a Widget expert; answer concisely with the facts."
SABOTAGE_PROMPT = "Answer with at most 5 words and do not use any specific numbers."


def run_eval(label, system_prompt):
    run_id = str(uuid.uuid4())
    rows = []
    scores = []
    for ex in EVAL_SLICE:
        out = model_under_test(ex["input"], system_prompt)
        s = judge_score(ex["input"], ex["reference_output"], out)
        scores.append(s)
        rows.append({"run_id": run_id, "example_id": ex["id"], "score": float(s)})
    supabase.table(RESULTS_TABLE).insert(rows).execute()
    mean = statistics.mean(scores)
    supabase.table(RUNS_TABLE).insert({"run_id": run_id, "label": label, "mean_score": float(mean)}).execute()
    return run_id, mean


baseline_run_id, baseline_mean = run_eval("baseline", BASELINE_PROMPT)
print(f"Baseline run {baseline_run_id} mean = {baseline_mean:.2f}")

# Persist baseline for the CI gate to pick up.
with open(BASELINE_PATH, "w") as f:
    json.dump({"run_id": baseline_run_id, "mean_score": baseline_mean, "label": "baseline"}, f)

sabotage_run_id, sabotage_mean = run_eval("sabotage", SABOTAGE_PROMPT)
print(f"Sabotage run {sabotage_run_id} mean = {sabotage_mean:.2f}")


def detect_regression(prev_score, curr_score, threshold=REGRESSION_THRESHOLD_POINTS):
    # YOUR CODE: return "block" if (prev_score - curr_score) >= threshold, else "allow".
    raise NotImplementedError("Replace with the threshold comparison.")


verdict = detect_regression(baseline_mean, sabotage_mean)
print(f"Detector verdict: {verdict} (delta = {baseline_mean - sabotage_mean:.2f} points)")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: sabotage run is at least 3 points below baseline; detector returns "block".


def checkpoint_3(session):
    if not os.path.exists(BASELINE_PATH):
        return CheckpointResult(status="fail", error_reason=f"{BASELINE_PATH} missing; baseline persistence failed.")
    runs = supabase.table(RUNS_TABLE).select("label,mean_score").order("id", desc=True).limit(2).execute().data
    if len(runs) < 2:
        return CheckpointResult(status="fail", error_reason=f"Expected >= 2 rows in {RUNS_TABLE}; got {len(runs)}.")
    by_label = {r["label"]: r["mean_score"] for r in runs}
    if "baseline" not in by_label or "sabotage" not in by_label:
        return CheckpointResult(
            status="fail",
            error_reason=f"Runs missing baseline/sabotage label; got {list(by_label.keys())}.",
        )
    delta = by_label["baseline"] - by_label["sabotage"]
    if delta < REGRESSION_THRESHOLD_POINTS:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Sabotage scored {by_label['sabotage']:.2f}, baseline {by_label['baseline']:.2f}; "
                f"delta = {delta:.2f} < {REGRESSION_THRESHOLD_POINTS}. The sabotage prompt may not be aggressive enough."
            ),
        )
    if verdict != "block":
        return CheckpointResult(
            status="fail",
            error_reason=f"detect_regression returned {verdict!r}; expected 'block' on a 3+ point drop.",
        )
    return CheckpointResult(status="pass")


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

`detect_regression` is one line: compare the delta to the threshold.

</details>

<details><summary>Hint 2 (direction)</summary>

```
def detect_regression(prev_score, curr_score, threshold=REGRESSION_THRESHOLD_POINTS):
    return "block" if (prev_score - curr_score) >= threshold else "allow"
```

If `prev_score - curr_score` is negative (the model got better), the function returns "allow"; that is correct behavior.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2.

</details>

**Wallet check.** 60 model-under-test calls + 60 judge calls at Sonnet rates: roughly $1.20. Cumulative: ~$1.25.

## Task 4: CI gate that fails on regression

Write a GitHub Actions workflow file to `/content/.github/workflows/eval.yml` that calls a Python script which reads the latest two runs and exits 1 on regression. Because the lab does not have access to a real GitHub runner, we ALSO write a local Python wrapper that simulates the workflow's exit-code check. The validator runs that wrapper and asserts exit code 1 when the sabotage run is the latest.

In [ ]:
# Task 4: workflow file + local simulator.

WORKFLOW_YAML = textwrap.dedent("""
name: Eval regression gate

on:
  pull_request:
    branches: [main]

jobs:
  eval-gate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - run: pip install supabase==2.9.0
      - run: python ci/eval_gate.py
        env:
          SUPABASE_URL: ${{ secrets.SUPABASE_URL }}
          SUPABASE_SERVICE_ROLE_KEY: ${{ secrets.SUPABASE_SERVICE_ROLE_KEY }}
""").strip()


def write_workflow():
    # YOUR CODE: ensure the parent directories of WORKFLOW_PATH exist
    # (os.makedirs with exist_ok=True), then write WORKFLOW_YAML to it.
    raise NotImplementedError("Replace with the path setup + write.")


write_workflow()
print(f"Workflow written to {WORKFLOW_PATH}")


# Validate the yaml parses.
parsed = yaml.safe_load(open(WORKFLOW_PATH))
assert "jobs" in parsed and "eval-gate" in parsed["jobs"], "workflow yaml malformed"
print("Workflow yaml parses.")


# Local simulator. Reads the latest two rows from the runs table, applies
# detect_regression, exits with 0 (allow) or 1 (block).
def run_ci_gate_locally():
    runs = supabase.table(RUNS_TABLE).select("label,mean_score").order("id", desc=True).limit(2).execute().data
    if len(runs) < 2:
        return 0  # nothing to compare, allow
    curr_score = runs[0]["mean_score"]
    prev_score = runs[1]["mean_score"]
    if detect_regression(prev_score, curr_score) == "block":
        return 1
    return 0


exit_code = run_ci_gate_locally()
print(f"Local CI gate exit code: {exit_code}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: workflow file exists, yaml parses, local simulator exits 1
# on the current state (sabotage is the latest run).


def checkpoint_4(session):
    if not os.path.exists(WORKFLOW_PATH):
        return CheckpointResult(status="fail", error_reason=f"{WORKFLOW_PATH} missing.")
    try:
        parsed = yaml.safe_load(open(WORKFLOW_PATH).read())
    except yaml.YAMLError as exc:
        return CheckpointResult(status="fail", error_reason=f"Workflow yaml does not parse: {exc!r}")
    if "jobs" not in parsed or "eval-gate" not in (parsed.get("jobs") or {}):
        return CheckpointResult(status="fail", error_reason="Workflow yaml missing jobs.eval-gate.")
    # The local simulator must report block (exit 1) on the current state.
    if exit_code != 1:
        return CheckpointResult(
            status="fail",
            error_reason=(
                f"Local CI gate exit code is {exit_code}; expected 1 because the sabotage run is the latest. "
                f"Check that the simulator orders runs DESC by id and applies detect_regression(prev, curr)."
            ),
        )
    return CheckpointResult(status="pass")


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The workflow file lives nested under `.github/workflows/`. You need `os.makedirs` with `exist_ok=True` before the write.

</details>

<details><summary>Hint 2 (direction)</summary>

```
def write_workflow():
    os.makedirs(os.path.dirname(WORKFLOW_PATH), exist_ok=True)
    open(WORKFLOW_PATH, "w").write(WORKFLOW_YAML)
```

</details>

<details><summary>Hint 3 (spoiler)</summary>

Same as Hint 2.

</details>

**Wallet check.** Zero new LLM calls. Cumulative: ~$1.25.

## Cleanup

Clear both Supabase tables, drop the workflow file and the baseline JSON. Tables stay (DROP requires direct Postgres access); the cleanup notes that. Re-running is safe.

In [ ]:
# NBVAL_SKIP
# Cleanup.

result = run_cleanup(CLEANUP_MANIFEST, adapter=CLEANUP_ADAPTER)

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids = set()
for warning_msg in result.warnings:
    for res in CLEANUP_MANIFEST:
        if res.id in warning_msg:
            failed_ids.add(res.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, Supabase tables may still hold rows. Resolve before closing.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: about $1.25-1.50.** Two eval passes plus the judge dominated; Sonnet at $3/$15 per 1M, ~30 examples x 2 passes x (model-under-test + judge). Supabase free tier covered the persistence. Tables cleared; workflow file + baseline JSON removed.

## Reflection

These are not graded. They are for you.

1. The judge scores on a 1-10 scale. The team is debating switching to pairwise (A vs B). What is one production scenario where pairwise is strictly better, and what is one where it is strictly worse?

2. The detector fails the PR on a 3-point drop. The exec wants 1-point drops to fail too. What is one false-positive risk of tightening the threshold, and how would you measure that risk before flipping the threshold?

## Exam-Style Practice

**Question 1 (judge calibration):**

An eval harness uses LLM-as-judge with Claude Sonnet on a 1-10 scale. The team notices the judge gives 8 to almost everything. What is the most likely cause?

A. Judge model too small.
B. Rubric lacks concrete examples per score level.
C. Reference outputs are bad.
D. Different judge model is needed.

<details><summary>Show answer</summary>

**Correct: B.**

- A is unlikely: Sonnet is more than capable as a judge; smaller judges (Haiku) usually go the other way and cluster at 5.
- B is correct: uncalibrated rubrics cluster at the middle-upper of any scale. The fix is 1-2 concrete examples per level in the rubric so the judge has anchors.
- C is rarely the cause of clustering: bad references usually produce bimodal scores (1 or 10), not a flat 8.
- D is wrong: the model is fine; the rubric is the lever.

</details>

**Question 2 (regression threshold):**

A team's eval harness fires "block" on every 3-point drop. The product owner wants to tighten to 1-point drops. What is the dominant risk?

A. Block rate climbs because LLM-judge noise produces 1-point drift on most A/B tests.
B. Block rate falls because the judge is too coarse.
C. The CI gate becomes faster.
D. The cost rises because the eval runs more often.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: LLM judges have run-to-run variance even on the same inputs; 1-point drift is below the noise floor of a typical 100-example eval. The detector starts firing on noise, the team loses trust, the gate gets disabled.
- B is wrong: tightening the threshold raises block rate, not falls.
- C is unrelated.
- D is unrelated; the threshold does not affect run frequency.

</details>

**Question 3 (golden-set drift):**

A team's golden set is 6 months old. The product surface area grew; new features are not in the set. What is the highest-leverage maintenance practice?

A. Replace the golden set every quarter with a fresh one.
B. Add 5-10 new examples per shipped feature; never remove old ones unless the feature is deprecated.
C. Run the eval against a larger model so the judge is more lenient.
D. Increase the threshold to allow more drift.

<details><summary>Show answer</summary>

**Correct: B.**

- A loses regression history; you cannot compare to last quarter's runs.
- B is correct: golden sets are append-only ledgers. The longer they live, the more confidence each run carries.
- C does not help with coverage; it hides regression noise.
- D is the bullet C; it weakens the gate to mask the coverage gap.

</details>